# Llama Evaluation via Groq

This notebook evaluates Llama models on the OP Benchmark using Groq's fast inference API.

**Install:** `pip install openai`

**Models available:**
- `llama-3.3-70b-versatile` - Best quality (recommended)
- `llama-3.1-8b-instant` - Faster, smaller
- `llama-3.2-90b-vision-preview` - Vision capable


In [ ]:
from openai import OpenAI

api_key = "YOUR_GROQ_API_KEY"  # ← Your Groq API key

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
import pandas as pd
import os
import glob

# ============ CONFIGURATION ============
BATCH_MODE = False  # Set to True to process multiple files
RATA_FOLDER = "Benchmark/4-Intermediate_1/Translations_GPT5.2"
RESULTS_FOLDER = "Results"

# Single file mode - main benchmark
input_csv_path = "Results/benchmark_results/llama_3.3_70b_OP_Benchmark.csv"

# Initialize for both modes
pending_files = []

if BATCH_MODE:
    all_files = glob.glob(os.path.join(RATA_FOLDER, "rata_100_*.csv"))
    all_files = sorted(all_files)
    
    processed_files = []
    
    print(f"Found {len(all_files)} RATA translation files\n")
    print("Status check:")
    print("-" * 80)
    
    for file_path in all_files:
        filename = os.path.splitext(os.path.basename(file_path))[0]
        output_csv = os.path.join(RESULTS_FOLDER, f"llama_3.3_70b_{filename}.csv")
        
        if os.path.exists(output_csv):
            try:
                output_df = pd.read_csv(output_csv)
                input_df = pd.read_csv(file_path)
                if len(output_df) == len(input_df) and output_df['answer_to_prompt_1'].notna().all() and output_df['answer_to_prompt_2'].notna().all():
                    processed_files.append(file_path)
                    print(f"✅ {filename} - COMPLETE ({len(output_df)} rows)")
                else:
                    pending_files.append(file_path)
                    print(f"🔄 {filename} - INCOMPLETE ({output_df['answer_to_prompt_1'].notna().sum()}/{len(output_df)} rows)")
            except:
                pending_files.append(file_path)
                print(f"❌ {filename} - ERROR reading output")
        else:
            pending_files.append(file_path)
            print(f"⚪ {filename} - NOT STARTED")
    
    print("-" * 80)
    print(f"\nSummary: {len(processed_files)} complete, {len(pending_files)} pending\n")
    
    if pending_files:
        print(f"Will process {len(pending_files)} files:")
        for f in pending_files:
            print(f"  - {os.path.basename(f)}")
        input_csv_path = pending_files[0]
        print(f"\n📂 Loading first pending file: {os.path.basename(input_csv_path)}")
    else:
        print("✅ All files already processed!")
        input_csv_path = None
else:
    print(f"📂 Single file mode: {os.path.basename(input_csv_path)}")
    pending_files = [input_csv_path]

# Load the data
if input_csv_path:
    df = pd.read_csv(input_csv_path)
    input_filename = os.path.splitext(os.path.basename(input_csv_path))[0]
    print(f"Loaded {len(df)} questions from {input_filename}")
    df.head()
else:
    df = None
    input_filename = None

In [ ]:
import time, threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import APIError, RateLimitError, APITimeoutError

# ============ CONFIGURATION ============
# Model options:
# - "llama-3.3-70b-versatile" (best quality - recommended)
# - "llama-3.1-8b-instant" (faster, smaller)
# - "llama-3.2-90b-vision-preview" (vision capable)

model_name = "llama-3.3-70b-versatile"

# Groq has generous limits but let's be safe
MAX_TOKENS = 16384  
MAX_WORKERS = 4
RPM_LIMIT = 30  # Groq free tier: 30 RPM
SAVE_EVERY = 10

# Output file name
CSV_PATH = os.path.join(RESULTS_FOLDER, f"llama_3.3_70b_Benchmark/OP_Benchmark.csv")

# ============ RATE LIMITER ============
class RPMLimiter:
    def __init__(self, rpm):
        self.rpm = rpm
        self.calls = deque()
        self.lock = threading.Lock()

    def wait(self):
        while True:
            with self.lock:
                now = time.time()
                while self.calls and now - self.calls[0] >= 60:
                    self.calls.popleft()
                if len(self.calls) < self.rpm:
                    self.calls.append(now)
                    return
            time.sleep(0.05)

rpm_limiter = RPMLimiter(RPM_LIMIT)
save_lock = threading.Lock()

# ============ API CALL ============
def ask_until_success(prompt):
    """Returns: (answer_text, input_tokens, output_tokens)"""
    while True:
        try:
            rpm_limiter.wait()
            
            response = client.chat.completions.create(
                model=model_name,
                max_tokens=MAX_TOKENS,
                messages=[{"role": "user", "content": prompt}]
            )
            
            text = ""
            if response.choices:
                message = response.choices[0].message
                text = (message.content or "").strip()
            
            input_tokens = 0
            output_tokens = 0
            
            if hasattr(response, 'usage') and response.usage:
                input_tokens = getattr(response.usage, 'prompt_tokens', 0) or 0
                output_tokens = getattr(response.usage, 'completion_tokens', 0) or 0
            
            if text:
                return text, input_tokens, output_tokens
            
            print("⛔ Empty response — retrying in 4s…")
            time.sleep(4)

        except RateLimitError as e:
            print(f"⚠️ Rate limit — retrying in 30s: {e}")
            time.sleep(30)
        except (APIError, APITimeoutError) as e:
            print(f"⚠️ API issue — retrying in 4s: {e}")
            time.sleep(4)
        except Exception as e:
            print(f"❌ Error — retrying in 4s: {e}")
            time.sleep(4)

# ============ PROCESS A SINGLE FILE ============
def process_file(file_path):
    """Process a single file and save results"""
    filename = os.path.splitext(os.path.basename(file_path))[0]
    
    if BATCH_MODE:
        csv_path = os.path.join(RESULTS_FOLDER, f"llama_3.3_70b_{filename}.csv")
    else:
        csv_path = CSV_PATH
    
    print(f"\n{'='*60}")
    print(f"📂 Processing: {filename}")
    print(f"📁 Output: {csv_path}")
    print(f"{'='*60}")
    
    file_df = pd.read_csv(file_path)
    
    if os.path.exists(csv_path):
        file_df = pd.read_csv(csv_path)
        print(f"📥 Loaded existing progress: {file_df['answer_to_prompt_1'].notna().sum()}/{len(file_df)} rows complete")
    
    # Add columns if missing
    for col in ['answer_to_prompt_1', 'answer_to_prompt_2', 
                'input_tokens_1', 'input_tokens_2',
                'output_tokens_1', 'output_tokens_2']:
        if col not in file_df.columns:
            file_df[col] = None
    
    file_df.reset_index(drop=True, inplace=True)
    
    def save_file():
        with save_lock:
            file_df.to_csv(csv_path, index=False)
    
    row_lock = threading.Lock()
    progress = {"count": 0}
    
    def process_row(i):
        if pd.notna(file_df.at[i, 'answer_to_prompt_1']) and pd.notna(file_df.at[i, 'answer_to_prompt_2']):
            return

        p1 = str(file_df.at[i, 'prompt.1'])
        p2 = str(file_df.at[i, 'prompt.2'])
        opt = str(file_df.at[i, 'OPTIONS'])

        if "OPTIONS" in p1: p1 = p1.replace("OPTIONS", opt)
        if "OPTIONS" in p2: p2 = p2.replace("OPTIONS", opt)

        a1, in1, out1 = ask_until_success(p1)
        a2, in2, out2 = ask_until_success(p2)

        with row_lock:
            file_df.at[i, 'answer_to_prompt_1'] = a1
            file_df.at[i, 'answer_to_prompt_2'] = a2
            file_df.at[i, 'input_tokens_1'] = in1
            file_df.at[i, 'input_tokens_2'] = in2
            file_df.at[i, 'output_tokens_1'] = out1
            file_df.at[i, 'output_tokens_2'] = out2
            progress["count"] += 1

            if progress["count"] % SAVE_EVERY == 0:
                save_file()
                print(f"💾 [{filename}] Saved @ {progress['count']} rows")
    
    todo = [i for i in range(len(file_df))
            if pd.isna(file_df.at[i, 'answer_to_prompt_1']) or pd.isna(file_df.at[i, 'answer_to_prompt_2'])]
    
    if not todo:
        print(f"✅ [{filename}] Already complete!")
        return
    
    print(f"🚀 [{filename}] {len(todo)} rows to process")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_row, i) for i in todo]
        for idx, _ in enumerate(as_completed(futures), 1):
            if idx % 10 == 0:
                print(f"[{filename}] Progress: {idx}/{len(todo)}")
    
    save_file()
    
    # Final verification
    missing = file_df[file_df['answer_to_prompt_1'].isna() | file_df['answer_to_prompt_2'].isna()].index.tolist()
    while missing:
        print(f"🔄 [{filename}] Fixing {len(missing)} missing...")
        for i in missing:
            process_row(i)
            save_file()
        missing = file_df[file_df['answer_to_prompt_1'].isna() | file_df['answer_to_prompt_2'].isna()].index.tolist()
    
    save_file()
    print(f"✅ [{filename}] COMPLETE — Saved to {csv_path}")

# ============ MAIN LOOP ============
if pending_files:
    print(f"\n🚀 Starting processing of {len(pending_files)} files...")
    print(f"🤖 Model: {model_name}")
    print(f"📊 Max tokens: {MAX_TOKENS}")
    
    for file_idx, file_path in enumerate(pending_files, 1):
        print(f"\n{'#'*60}")
        print(f"# FILE {file_idx}/{len(pending_files)}")
        print(f"{'#'*60}")
        process_file(file_path)
    
    print(f"\n{'='*60}")
    print(f"🎉 ALL {len(pending_files)} FILES PROCESSED!")
    print(f"{'='*60}")
else:
    print("✅ No pending files to process!")